# Dependencies
#### Run the following file(s) before running this code.
- 03_baseline_similarity_graph.ipynb (or 03b or 03c)
- 07_gds_Louvain_Summary.ipynb

##### Note:
URL of the Neo4j browser:
- https://[IP address]:7473/browser/

ID & Pass: 
- Use the one in .env


In [1]:
# Config
SAMPLING:bool       = True
NUM_SAMPLE:int      = 250   # Number of sample data to be ingested to the graph database
SUMMARY_SAMPLE:int  = 20    # Number of samples as inputs of summarizing
RAND_SEED:int       = 77    # Seed for sampling
NUM_SIM:int         = 5     # Number of results from KNN search (does not include the own node)
EMBEDDING_MODEL:str = "text-embedding-3-small"
MAX_TOKENS:int      = 7800  # Max 8192 - some safety buffer about 5%
INDEX_NAME:str      = "idx:complaints_vss"
FILE_PATH:str       = "../../data/original/complaints-2025-11-02_04_18.csv"

In [2]:
import time
from datetime import datetime, timedelta

In [3]:
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display
import tiktoken

In [4]:
from dotenv import load_dotenv  
load_dotenv()

True

In [5]:
import neo4j

In [6]:
# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

In [7]:
warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [8]:
# Show all columns
pd.set_option('display.max_columns', None)

# Show all rows
pd.set_option('display.max_rows', None)

In [9]:
# Timestamp (Start)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

start_time = time.time()

2026-01-08_22:02:23


### Neo4j

In [10]:
driver = neo4j.GraphDatabase.driver(
    uri=os.environ.get("NEO4J_URI"), 
    auth=(os.environ.get("NEO4J_USERNAME"), 
          os.environ.get("NEO4J_PASSWORD"))
)

In [11]:
session = driver.session(database="neo4j")

In [12]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

# Degree Centrality

In [13]:
# Drop the in-memory graph
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

# Create an in-memory graph named 'ds_graph'
# Only include 'Category' and 'Complaint' nodes and 'IN_CATEGORY' relationship
query = """
CALL gds.graph.project(
    'ds_graph', 
    ['Category', 'Complaint'], 
    {
        IN_CATEGORY: {
            type: 'IN_CATEGORY', 
            orientation: 'NATURAL'
        }
    }
)
"""
session.run(query)

In [14]:
# Note: 'UNDIRECTED': Count both in and out going arrows
query = """
CALL gds.degree.stream(
    'ds_graph',
    {orientation: 'UNDIRECTED'}
)
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
WHERE n:Category
RETURN n.summary AS Category, score AS Degree
ORDER BY Degree DESC, Category;
"""

my_neo4j_run_query_pandas(query).head(10)
# session.run(query)

,Category,Degree
0,Consumer complaints about unverified debts and...,106.0
1,Consumer complaints regarding unauthorized acc...,29.0
2,Consumer complaints about erroneous credit rep...,28.0
3,Consumer complaints about inadequate responses...,26.0
4,Consumer complaints about account access issue...,13.0
5,Consumer complaints about inadequate fraud pro...,11.0
6,Consumer complaints regarding mishandled fraud...,9.0
7,Consumer complaints about inaccessible account...,7.0
8,"Consumer complaints about account freezes, clo...",5.0
9,Consumer complaints about unexpected changes t...,4.0


# Harmonic Centrality

In [15]:
# Drop the in-memory graph
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

# Create an in-memory graph named 'ds_graph'
# Only include 'Category' and 'Complaint' nodes and 'IN_CATEGORY' relationship
query = """
CALL gds.graph.project(
    'ds_graph', 
    ['Category', 'Complaint'], 
    {
        IN_CATEGORY: {
            type: 'IN_CATEGORY', 
            orientation: 'NATURAL'
        }
    }
)
"""

session.run(query)

In [16]:
query = """

CALL gds.closeness.harmonic.stream('ds_graph', {})
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
WHERE n:Category
RETURN n.summary AS Category, score as Closeness
ORDER BY Closeness DESC

"""

my_neo4j_run_query_pandas(query).head(10)

# Note: 
# This is Harmonic Centrality based on Category nodes and Complaint nodes.
# Considering all Complaint nodes are directly connected to Category nodes (by the design of this graph),
# the Closeness score in here has almost the same meaning as the Centrality analysis above

,Category,Closeness
0,Consumer complaints about unverified debts and...,0.401515
1,Consumer complaints regarding unauthorized acc...,0.109848
2,Consumer complaints about erroneous credit rep...,0.106061
3,Consumer complaints about inadequate responses...,0.098485
4,Consumer complaints about account access issue...,0.049242
5,Consumer complaints about inadequate fraud pro...,0.041667
6,Consumer complaints regarding mishandled fraud...,0.034091
7,Consumer complaints about inaccessible account...,0.026515
8,"Consumer complaints about account freezes, clo...",0.018939
9,Consumer complaints about unexpected changes t...,0.015152


# Analysis based on category nodes (without having complaint nodes)

In [17]:
# Drop the in-memory graph
query = "CALL gds.graph.drop('category_graph', false) yield graphName"
session.run(query)

# Step 1: Create an in-memory graph named 'category_graph'
          
query = """

MATCH (ca:Category)<-[:IN_CATEGORY]-(c1:Complaint)
    -[s:SIMILAR]- 
    (c2:Complaint)-[:IN_CATEGORY]->(cb:Category)
    
WHERE elementId(ca) < elementId(cb)   // avoid double counting and self-loops

WITH 
    ca AS source, 
    cb AS target, 
    avg(toFloat(s.similarity_score)) AS similarity

WITH gds.graph.project(
    'category_graph',
    source,    // source node
    target,    // target node
    {
        relationshipProperties: {
            similarity: similarity
        }
    }
) AS g

RETURN
    g.graphName  AS graph,
    g.nodeCount  AS nodes,
    g.relationshipCount AS rels;
"""

session.run(query)

In [18]:
# Step 2: Run Harmonic similarity
          
query = """
CALL gds.closeness.harmonic.stream('category_graph', {})
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).summary AS category, score
ORDER BY score DESC;
"""

my_neo4j_run_query_pandas(query).head(10)


# Note:
# Meaning of scores
# 1: Category is connected to exactly one other category with distance = 1
# 0.5: Category is connected to one other category with distance = 2
# Fraction (e.g., 0.428): Category is connected to multiple categories
# 0.0: Category is isolated

,category,score
0,Consumer complaints about account access issue...,0.821429
1,Consumer complaints about unexpected changes t...,0.642857
2,Consumer complaints regarding mishandled fraud...,0.607143
3,Consumer complaints about inaccessible funds a...,0.607143
4,Consumer complaints about inadequate responses...,0.535714
5,Consumer complaints about excessive holds on d...,0.500000
6,Consumer complaints about unfulfilled payment ...,0.488095
7,Consumer complaints about inaccessible account...,0.357143
8,Consumer complaints about unverified debts and...,0.285714
9,"Consumer complaints about account freezes, clo...",0.285714


# PageRank analysis

In [19]:
query = """
CALL gds.pageRank.stream('category_graph', {
  relationshipWeightProperty: 'similarity',
  dampingFactor: 0.85,
  maxIterations: 50
})
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).summary AS category, score
ORDER BY score DESC;
"""

my_neo4j_run_query_pandas(query).head(10)

,category,score
0,Consumer complaints about account access issue...,0.834881
1,Consumer complaints about unexpected changes t...,0.528112
2,Consumer complaints about inaccessible funds a...,0.438677
3,Consumer complaints regarding mishandled fraud...,0.426475
4,Consumer complaints about excessive holds on d...,0.382428
5,Consumer complaints about inadequate responses...,0.336294
6,Consumer complaints about unverified debts and...,0.289089
7,Consumer complaints about unfulfilled payment ...,0.265140
8,Consumer complaints about inaccessible account...,0.234562
9,"Consumer complaints about account freezes, clo...",0.203890


In [20]:
# Timestamp (End)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

end_time = time.time()
elapsed_seconds = end_time - start_time

# Convert elapsed seconds to minutes and seconds
minutes = int(elapsed_seconds // 60)
seconds = elapsed_seconds % 60

print(f"Program elapsed time: {minutes} minutes and {seconds:.2f} seconds")

2026-01-08_22:02:23
Program elapsed time: 0 minutes and 0.21 seconds
